# Experiment: MoNuSAC Annotation QC from a Single Image Path

Objective:
- Point the notebook at one exported RGB patch.
- Infer and load the matching instance-label mask automatically.
- Visualize the original overlay, then rescale the patch from 40x to 20x and compare the result.

Notes:
- Set `INPUT_IMAGE_PATH` in the configuration cell below.
- Leave `INPUT_MASK_PATH = None` when the mask lives next to the image and follows the usual `*_mask.png` naming.
- Relative paths are resolved from `scripts/benchmarking/` first, then the current working directory.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_notebook_dir(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / 'monusac_visualization_utils.py').exists():
            return candidate
        benchmark_dir = candidate / 'scripts' / 'benchmarking'
        if (benchmark_dir / 'monusac_visualization_utils.py').exists():
            return benchmark_dir

    hpc_dir = Path('/share/lab_teng/trainee/tusharsingh/cell-seg/scripts/benchmarking')
    if (hpc_dir / 'monusac_visualization_utils.py').exists():
        return hpc_dir

    raise FileNotFoundError('Could not locate scripts/benchmarking/monusac_visualization_utils.py.')


NOTEBOOK_DIR = find_notebook_dir()

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from monusac_visualization_utils import (
    DEFAULT_MIN_INSTANCE_FRACTION,
    default_monusac_root,
    load_sample_from_image_path,
    plot_resize_comparison,
    plot_sample_triptych,
    rescale_patch_and_mask,
)

PROJECT_ROOT = (NOTEBOOK_DIR / '../..').resolve()
DEFAULT_DATA_ROOT = default_monusac_root(search_from=NOTEBOOK_DIR)

print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Example MoNuSAC root: {DEFAULT_DATA_ROOT}')


## Input Path

Set `INPUT_IMAGE_PATH` to the RGB patch you want to inspect.
The notebook will infer the matching `*_mask.png` file automatically.
Use `INPUT_MASK_PATH` only if the mask is stored somewhere else or uses a different name.


In [ ]:
INPUT_IMAGE_PATH = "../../data/Monusac/all_merged/0009_train_0009_TCGA-EV-5903_kidney_image.png"
INPUT_MASK_PATH = None

SOURCE_MAGNIFICATION = 40
TARGET_MAGNIFICATION = 20
MIN_INSTANCE_FRACTION = DEFAULT_MIN_INSTANCE_FRACTION

print(f'Input image path: {INPUT_IMAGE_PATH}')
print(f'Input mask override: {INPUT_MASK_PATH}')


In [ ]:
sample_row, image, mask = load_sample_from_image_path(
    INPUT_IMAGE_PATH,
    mask_path=INPUT_MASK_PATH,
    search_from=NOTEBOOK_DIR,
)

mask_labels = np.unique(mask)
original_instance_count = int(mask_labels[mask_labels > 0].size)

print(f"Loaded sample: {sample_row['unique_id']}")
print(f"Resolved image path: {sample_row['image_path']}")
print(f"Resolved mask path: {sample_row['mask_path']}")
print(f'Image shape: {image.shape}')
print(f'Mask shape: {mask.shape}')
print(f'Instance count: {original_instance_count}')

pd.Series(sample_row)


In [ ]:
_ = plot_sample_triptych(
    image,
    mask,
    title=f"{sample_row['unique_id']} | original exported patch",
)


## 40x to 20x Rescaling

The RGB patch is downsampled with Pillow's `BOX` resampler, which behaves like area averaging for shrink operations.
The instance mask is resized one label at a time:
- convert each instance to a binary mask,
- downsample the binary mask with the same area-style resampler,
- assign each target pixel to the label with the strongest fractional support above `MIN_INSTANCE_FRACTION`.

This is safer than resizing the integer label map directly because label IDs are categorical, not continuous values.


In [ ]:
image_20x, mask_20x, resize_meta = rescale_patch_and_mask(
    image=image,
    instance_mask=mask,
    source_magnification=SOURCE_MAGNIFICATION,
    target_magnification=TARGET_MAGNIFICATION,
    min_instance_fraction=MIN_INSTANCE_FRACTION,
)

pd.Series(resize_meta)


In [ ]:
_ = plot_resize_comparison(
    original_image=image,
    original_mask=mask,
    resized_image=image_20x,
    resized_mask=mask_20x,
    title=f"{sample_row['unique_id']} | {SOURCE_MAGNIFICATION}x to {TARGET_MAGNIFICATION}x comparison",
)


## Minimal Reuse Snippet

Copy the call below into another notebook or preprocessing script if you want the same direct path-based QC workflow elsewhere in the project.


In [ ]:
sample_row, image, mask = load_sample_from_image_path(
    INPUT_IMAGE_PATH,
    mask_path=INPUT_MASK_PATH,
    search_from=NOTEBOOK_DIR,
)

rescaled_image, rescaled_mask, resize_metadata = rescale_patch_and_mask(
    image=image,
    instance_mask=mask,
    source_magnification=SOURCE_MAGNIFICATION,
    target_magnification=TARGET_MAGNIFICATION,
    min_instance_fraction=MIN_INSTANCE_FRACTION,
)
